# SessionRerank+ · Karnataka Smallholder Agriculture (v16 — adaptive + install fix)
### *Co-activation cached tool retrieval, powered by Gemma 4*

**v16 changes:** uninstall conflicting cu128 torchaudio first, force numpy<2 reinstall after deps, install repo as editable package so subprocesses find sessionrerank.*


## 1. Detect device + compute capability

In [ ]:
import os, subprocess, platform
def _gpu_info():
    try:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=10)
        if out.returncode != 0 or not out.stdout.strip():
            return ('cpu', 0, 0, 'cpu')
        name, cc = [x.strip() for x in out.stdout.strip().splitlines()[0].split(',')]
        maj, mn = cc.split('.')
        return ('cuda', int(maj), int(mn), name)
    except Exception as e:
        print('nvidia-smi probe failed:', e); return ('cpu', 0, 0, 'cpu')
DEVICE, CC_MAJOR, CC_MINOR, GPU_NAME = _gpu_info()
CC = CC_MAJOR * 10 + CC_MINOR
print(f'Device:      {DEVICE}')
print(f'GPU name:    {GPU_NAME}')
print(f'Compute cap: {CC_MAJOR}.{CC_MINOR}  (CC code = {CC})')
print(f'Python:      {platform.python_version()}')
if DEVICE == 'cpu':       DTYPE_STR = 'fp32'
elif CC < 70:             DTYPE_STR = 'fp32'
elif CC < 80:             DTYPE_STR = 'fp16'
else:                     DTYPE_STR = 'bf16'
print(f'Selected dtype path: {DTYPE_STR}')
os.environ['SR_DEVICE'] = DEVICE
os.environ['SR_DTYPE']  = DTYPE_STR
os.environ['SR_CC']     = str(CC)
os.environ['SR_GPU']    = GPU_NAME


## 2. Install torch compatible with the detected GPU
Uninstall cu128 family FIRST, then install device-appropriate wheel, then pin numpy<2 (without dependency resolution).

In [ ]:
import subprocess, sys, os
DEVICE = os.environ['SR_DEVICE']
CC = int(os.environ['SR_CC'])
def run(*c, check=False):
    print('+', ' '.join(c), flush=True)
    p = subprocess.Popen(list(c), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line.rstrip(), flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'cmd failed: {c}')
    return p.returncode

# Step 1: uninstall ALL preinstalled torch family
run(sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
    'torchaudio', 'torchvision', 'torch')

# Step 2: install device-appropriate wheel
if DEVICE == 'cpu':
    idx = 'https://download.pytorch.org/whl/cpu'
elif CC < 70:
    idx = 'https://download.pytorch.org/whl/cu118'
else:
    idx = 'https://download.pytorch.org/whl/cu121'
run(sys.executable, '-m', 'pip', 'install', '-q',
    '--index-url', idx, 'torch==2.4.1', 'torchvision==0.19.1', check=True)

# Step 3: force numpy<2 (no-deps so it doesn't drag anything else)
run(sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps', 'numpy==1.26.4', check=True)

# Step 4: smoke test
import importlib, sys as _s
for mod in ('numpy', 'torch'):
    if mod in _s.modules:
        del _s.modules[mod]
import numpy as np
import torch
print('numpy:', np.__version__, '(want 1.26.4)')
print('torch:', torch.__version__, 'cuda?', torch.cuda.is_available())
if torch.cuda.is_available():
    x = torch.randn(8, 8, device='cuda', dtype=torch.float32)
    y = (x @ x.T).sum().item()
    print('cuda smoke test OK; matmul sum =', y)


## 3. Clone repo, install lightweight deps + the repo as editable package

In [ ]:
import subprocess, sys, pathlib, os
def run(*c, check=False):
    print('+', ' '.join(c), flush=True)
    p = subprocess.Popen(list(c), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line.rstrip(), flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'cmd failed: {c}')
    return p.returncode

run('git', 'clone', '--depth', '1',
    'https://github.com/joyjeni/session-aware-toolbench-rerank.git',
    '/kaggle/working/sr', check=True)

run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45', 'sentence-transformers>=3.0',
    'huggingface_hub>=0.24', 'pandas', 'scikit-learn',
    'rank-bm25', 'tqdm', 'matplotlib')

# Re-pin numpy<2 after transformers install (it may have pulled numpy 2.x)
run(sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps', 'numpy==1.26.4', check=True)

# Install the repo as an editable package so subprocesses can `python -m sessionrerank...`
repo = pathlib.Path('/kaggle/working/sr')
if not ((repo/'pyproject.toml').exists() or (repo/'setup.py').exists()):
    (repo/'setup.py').write_text(
        "from setuptools import setup, find_packages\n"
        "setup(name='sessionrerank', version='0.1.0',\n"
        "      package_dir={'': 'src'}, packages=find_packages(where='src'),\n"
        "      python_requires='>=3.8')\n")
    print('Wrote', repo/'setup.py')
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo), check=True)

os.chdir('/kaggle/working/sr')
sys.path.insert(0, '/kaggle/working/sr/src')
print('Repo @', pathlib.Path('.').resolve())
run('ls')

# Verify the import works in BOTH this process AND a subprocess
import importlib
try:
    importlib.import_module('sessionrerank')
    print('sessionrerank package (this process): importable ✓')
except Exception as e:
    print('sessionrerank package (this process): FAILED:', e)
run(sys.executable, '-c', 'import sessionrerank; print("sessionrerank package (subprocess):", sessionrerank.__file__)')


## 4. HF token (optional)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN injected.')
except Exception as e:
    print('No HF_TOKEN secret — Gemma fine-tuned branch will be skipped.')


## 5. Download ToolBench splits

In [ ]:
import subprocess, sys
p = subprocess.Popen(
    [sys.executable, '-u', '-m', 'sessionrerank.data.download',
     '--out', '/kaggle/working/sr/data'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line.rstrip(), flush=True)
p.wait()
print('download exit:', p.returncode)


## 6. Run the cascade (live unbuffered output)

In [ ]:
import subprocess, sys, time, os
DEVICE = os.environ['SR_DEVICE']
DTYPE  = os.environ['SR_DTYPE']
cmd = [sys.executable, '-u', '-m', 'sessionrerank.eval.run_all',
       '--device', DEVICE,
       '--dtype',  DTYPE,
       '--seeds',  '0,1,2,42,123',
       '--pheromone',
       '--pheromone-rho',   '0.02',
       '--pheromone-tau0',  '1.0',
       '--pheromone-decay', '0.7',
       '--cooc-window',     '3',
       '--cross-encoder',
       '--checkpoint-every-method']
if DEVICE == 'cuda':
    cmd += ['--gemma-rerank', 'google/gemma-2-2b-it']
else:
    cmd += ['--skip-gemma-rerank']
print('+', ' '.join(cmd), flush=True)
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line.rstrip(), flush=True)
p.wait()
print(f'\n[cascade] exit code = {p.returncode}   elapsed = {time.time()-t0:.1f}s', flush=True)


## 7. Load + display metrics

In [ ]:
import json, pathlib, pandas as pd
p = pathlib.Path('results/metrics.json')
if not p.exists():
    print('No metrics.json yet (cascade did not complete).')
else:
    m = json.loads(p.read_text())
    rows = []
    for method, splits in m.items():
        row = {'method': method}
        if isinstance(splits, dict):
            for split, vals in splits.items():
                if isinstance(vals, dict):
                    row[f'{split}_NDCG@5'] = vals.get('NDCG@5')
                    row[f'{split}_Hit@5']  = vals.get('Hit@5')
        rows.append(row)
    df = pd.DataFrame(rows).set_index('method')
    print(df.to_string())
    df.to_csv('results/metrics_summary.csv')
    print('\nWrote results/metrics_summary.csv')


## 8. Karnataka first-hit accuracy

In [ ]:
import sys, json, pathlib
sys.path.insert(0, '/kaggle/working/sr/agri_extension')
try:
    from sessionrerank_plus import SessionRerankPlus, FarmerProfile
    from karnataka_apis import KARNATAKA_APIS
except Exception as e:
    print('Karnataka extension import failed:', e)
    SessionRerankPlus = None

if SessionRerankPlus is not None:
    ranker = SessionRerankPlus()
    bank = [
        ('My ragi crop has yellowing leaves and dry patches', 'ksda_pest_alert'),
        ('When can I claim crop insurance for ragi this kharif', 'pmfby'),
        ('What is today ragi price at Tumakuru mandi', 'agmarknet_ka'),
        ('Need helpline to ask about NPK dose for ragi', 'k_kisan'),
        ('Check my Bhoomi land record for survey number 123/4', 'bhoomi'),
        ('Is there a heatwave warning for Tumakuru', 'ksndmc'),
        ('Soil health card request for my farm', 'soil_health_card'),
        ('PM-KISAN installment status', 'pm_kisan'),
        ('Milk procurement rate today at KMF Tumkur dairy', 'kmf_nandini'),
        ('Offline pest identification for whitefly on tomato', 'plantix_uasb'),
    ]
    profile = FarmerProfile(crop='ragi', district='Tumakuru', language='kn', season='kharif')
    ranker.set_profile(profile)
    hits = 0
    per_query = []
    for q, gold in bank:
        ranked = ranker.rank(q, top_k=1)
        top = ranked[0]['api_id']
        hit = top == gold
        hits += int(hit)
        per_query.append({'query': q, 'gold': gold, 'top': top, 'hit': hit})
        print(f"{'OK' if hit else '..'}  {q[:48]:48}  top={top:18}  gold={gold}", flush=True)
    acc = hits / len(bank)
    print(f'\nFirst-hit accuracy: {hits}/{len(bank)} = {acc:.2f}')
    mp = pathlib.Path('results/metrics.json')
    m = json.loads(mp.read_text()) if mp.exists() else {}
    m.setdefault('SessionRerank+', {}).setdefault('karnataka', {})['first_hit'] = acc
    m['SessionRerank+']['karnataka']['per_query'] = per_query
    mp.parent.mkdir(parents=True, exist_ok=True)
    mp.write_text(json.dumps(m, indent=2))
    print('Wrote', mp)


## 9. Citation
```
@misc{sessionrerank2026,
  title  = {SessionRerank+: A Session-Aware Re-Ranking for API Tool Retrieval in Agentic Large Language Model Systems},
  author = {Jenisha T and Jyothi A P},
  year   = {2026},
  url    = {https://github.com/joyjeni/session-aware-toolbench-rerank}
}
```
